In [ ]:
import pandas as pd
import numpy as np

df=pd.read_json('regex_output.ndjson',lines=True)
df.head(2)

,source,raw_log,clean_message,log_level,service,timestamp,structured_fields,label,confidence,classified_by
0,mac,Jul 1 09:00:55 calvisitor-10-105-160-95 kerne...,iothunderboltswitch<<NUM>>(0x0)::listenercallb...,INFO,kernel,2026-07-01 09:00:55,"{'host': 'calvisitor-10-105-160-95', 'pid': '0...",None,0.0,Unclassified
1,mac,Jul 1 09:01:05 calvisitor-10-105-160-95 com.a...,thermal pressure state: <NUM> memory pressure ...,WARN,com.apple.CDScheduler,2026-07-01 09:01:05,"{'host': 'calvisitor-10-105-160-95', 'pid': '43'}",None,0.0,Unclassified


In [ ]:
df.classified_by.value_counts()

,count
classified_by,
Unclassified,8670
regex,1330


In [ ]:
from sentence_transformers import SentenceTransformer

class BertEmbedder:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def embed(self, texts):
        """
        texts: List[str]
        returns: numpy array of embeddings
        """
        return self.model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=False
        )


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

class BertEngine:
    def __init__(self, embedder, similarity_threshold=0.80):
        self.embedder = embedder
        self.similarity_threshold = similarity_threshold

        # Memory of known embeddings
        self.known_embeddings = []
        self.known_labels = []

    def add_known_examples(self, texts, labels):
        """
        texts: List[str]
        labels: List[str]
        """
        embeddings = self.embedder.embed(texts)
        self.known_embeddings.extend(embeddings)
        self.known_labels.extend(labels)

    def classify(self, text):
        """
        Classify a single Unclassified log
        """
        if not self.known_embeddings:
            return {
                "label": None,
                "confidence": 0.0,
                "classified_by": "bert"
            }

        query_embedding = self.embedder.embed([text])[0]

        sims = cosine_similarity(
            [query_embedding],
            self.known_embeddings
        )[0]

        best_idx = int(np.argmax(sims))
        best_score = float(sims[best_idx])

        if best_score >= self.similarity_threshold:
            return {
                "label": self.known_labels[best_idx],
                "confidence": best_score,
                "classified_by": "bert"
            }

        return {
            "label": None,
            "confidence": best_score,
            "classified_by": "bert"
        }


In [ ]:
import json

INPUT = "./regex_output.ndjson"
OUTPUT = "./bert_output.ndjson"

embedder = BertEmbedder()
engine = BertEngine(embedder)

# Step 1: Load known labeled logs
known_texts = []
known_labels = []

with open(INPUT) as f:
    for line in f:
        log = json.loads(line)
        if log["label"] and log["classified_by"] == "regex":
            known_texts.append(log["clean_message"])
            known_labels.append(log["label"])

engine.add_known_examples(known_texts, known_labels)

# Step 2: Classify unclassified logs
with open(INPUT) as fin, open(OUTPUT, "w") as fout:
    for line in fin:
        log = json.loads(line)

        if log["classified_by"] == "Unclassified":
            result = engine.classify(log["clean_message"])
            log.update(result)

        fout.write(json.dumps(log) + "\n")

print("Phase 2 completed.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Phase 2 completed.


In [ ]:
import pandas as pd
df=pd.read_json('bert_output.ndjson',lines=True)
df.head()

,source,raw_log,clean_message,log_level,service,timestamp,structured_fields,label,confidence,classified_by
0,mac,Jul 1 09:00:55 calvisitor-10-105-160-95 kerne...,iothunderboltswitch<<NUM>>(0x0)::listenercallb...,INFO,kernel,2026-07-01 09:00:55,"{'host': 'calvisitor-10-105-160-95', 'pid': '0...",None,0.250470,bert
1,mac,Jul 1 09:01:05 calvisitor-10-105-160-95 com.a...,thermal pressure state: <NUM> memory pressure ...,WARN,com.apple.CDScheduler,2026-07-01 09:01:05,"{'host': 'calvisitor-10-105-160-95', 'pid': '43'}",None,0.372277,bert
2,mac,Jul 1 09:01:06 calvisitor-10-105-160-95 QQ[10...,fa||url||taskid[<NUM>] dealloc,INFO,QQ,2026-07-01 09:01:06,"{'host': 'calvisitor-10-105-160-95', 'pid': '1...",None,0.339929,bert
3,mac,Jul 1 09:02:26 calvisitor-10-105-160-95 kerne...,arpt: <NUM>.<NUM>: airport_brcm43xx::syncpower...,INFO,kernel,2026-07-01 09:02:26,"{'host': 'calvisitor-10-105-160-95', 'pid': '0...",None,0.396106,bert
4,mac,Jul 1 09:02:26 authorMacBook-Pro kernel[0]: A...,arpt: <NUM>.<NUM>: airport_brcm43xx::platformw...,INFO,kernel,2026-07-01 09:02:26,"{'host': 'authorMacBook-Pro', 'pid': '0', 'com...",None,0.394218,bert


In [ ]:
df.classified_by.value_counts()

,count
classified_by,
bert,8670
regex,1330


In [ ]:
df[df['confidence'] >= .75].shape

(1926, 10)

In [ ]:
from sentence_transformers import SentenceTransformer

class BertEmbedder:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)

    def encode(self, texts):
        return self.model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=False
        )
import joblib
import numpy as np

class MLEngine:
    def __init__(self, model_path, threshold=0.80):
        self.model = joblib.load(model_path)
        self.embedder = BertEmbedder()
        self.threshold = threshold

    def classify(self, text):
        embedding = self.embedder.encode([text])
        probs = self.model.predict_proba(embedding)[0]

        best_idx = int(np.argmax(probs))
        best_prob = float(probs[best_idx])
        best_label = self.model.classes_[best_idx]

        if best_prob >= self.threshold:
            return {
                "label": best_label,
                "confidence": best_prob,
                "classified_by": "ml"
            }

        return {
            "label": None,
            "confidence": best_prob,
            "classified_by": "ml"
        }


In [ ]:
import json
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

INPUT = "./regex_output.ndjson"
MODEL_PATH = "./log_classifier.joblib"

texts, labels = [], []

with open(INPUT) as f:
    for line in f:
        log = json.loads(line)
        if (log["label"]!= "" ) and log["classified_by"] != "regex":
            texts.append(log["clean_message"])
            labels.append(log["label"])

print(f"[INFO] Training samples: {len(texts)}")

embedder = BertEmbedder()
X = embedder.encode(texts)
y = labels

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42,
    stratify=y
)

y_train = le.fit_transform(y_train)
y_val = le.transform(y_val)

clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)

clf.fit(X_train, y_train)

print(classification_report(y_val, clf.predict(X_val)))

joblib.dump(clf, MODEL_PATH)
print("[DONE] Model trained and saved")


[INFO] Training samples: 8670


TypeError: '<' not supported between instances of 'NoneType' and 'NoneType'

In [ ]:
import json
import numpy as np
import joblib
from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

# Assuming embedder is already initialized:
# embedder = BertEmbedder()

le = LabelEncoder()

INPUT = "./regex_output.ndjson"
MODEL_PATH = "./log_classifier.joblib"

texts, labels = [], []

print("[INFO] Loading data...")

with open(INPUT) as f:
    for line in f:
        try:
            log = json.loads(line)

            # Safe extraction
            label_val = str(log.get("label") or "").strip()
            message = log.get("clean_message", "").strip()

            # --- FIX IS HERE ---
            # We removed 'and log.get("classified_by") != "regex"'
            # Now we accept ANY valid label, regardless of source.
            if label_val != "" and message != "":
                texts.append(message)
                labels.append(label_val)

        except json.JSONDecodeError:
            continue

# 1. Validation
label_counts = Counter(labels)
print(f"[INFO] Samples loaded: {len(texts)}")
print(f"[INFO] Label distribution: {dict(label_counts)}")

if len(label_counts) < 2:
    raise ValueError(f"Still not enough classes. Found: {list(label_counts.keys())}")

# 2. Vectorization
print("[INFO] Encoding texts...")
X = embedder.encode(texts)
y = np.array(labels)

# 3. Stratified Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Encoding Targets
y_train_encoded = le.fit_transform(y_train)
y_val_encoded = le.transform(y_val)

# 5. Training
print("[INFO] Training model...")
clf = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    n_jobs=-1
)
clf.fit(X_train, y_train_encoded)

# 6. Evaluation
print("\n--- Classification Report ---")
preds = clf.predict(X_val)
print(classification_report(y_val_encoded, preds, target_names=le.classes_))

# 7. Save
# joblib.dump({"model": clf, "encoder": le}, MODEL_PATH)
# print(f"[DONE] Saved model to {MODEL_PATH}")

[INFO] Loading data...
[INFO] Samples loaded: 1330
[INFO] Label distribution: {'AUTH_FAILURE': 490, 'UNKNOWN_USER_LOGIN': 117, 'HDFS_BLOCK_STORED': 314, 'HDFS_BLOCK_RECEIVED': 294, 'HDFS_BLOCK_ALLOCATED': 115}
[INFO] Encoding texts...
[INFO] Training model...

--- Classification Report ---
                      precision    recall  f1-score   support

        AUTH_FAILURE       1.00      1.00      1.00        98
HDFS_BLOCK_ALLOCATED       1.00      1.00      1.00        23
 HDFS_BLOCK_RECEIVED       1.00      1.00      1.00        59
   HDFS_BLOCK_STORED       1.00      1.00      1.00        63
  UNKNOWN_USER_LOGIN       1.00      1.00      1.00        23

            accuracy                           1.00       266
           macro avg       1.00      1.00      1.00       266
        weighted avg       1.00      1.00      1.00       266

[DONE] Saved model to ./log_classifier.joblib


In [ ]:
import joblib
import numpy as np
from embedder import BertEmbedder

class MLEngine:
    def __init__(self, model_path, threshold=0.80):
        self.model = joblib.load(model_path)
        self.embedder = BertEmbedder()
        self.threshold = threshold

    def classify(self, text):
        embedding = self.embedder.encode([text])
        probs = self.model.predict_proba(embedding)[0]

        best_idx = int(np.argmax(probs))
        best_prob = float(probs[best_idx])
        best_label = self.model.classes_[best_idx]

        if best_prob >= self.threshold:
            return {
                "label": best_label,
                "confidence": best_prob,
                "classified_by": "ml"
            }

        return {
            "label": None,
            "confidence": best_prob,
            "classified_by": "ml"
        }


In [ ]:
import json
from ml.ml_engine import MLEngine

INPUT = "regex_output.ndjson"
OUTPUT = "bert_output.ndjson"

engine = MLEngine("ml/log_classifier.joblib")

with open(INPUT) as fin, open(OUTPUT, "w") as fout:
    for line in fin:
        log = json.loads(line)

        if log["classified_by"] == "Unclassified":
            result = engine.classify(log["clean_message"])
            log.update(result)

        fout.write(json.dumps(log) + "\n")

print("[DONE] Phase 2 ML inference complete")


In [56]:
df[df.label=='APACHE_WORKER_INIT']

,source,raw_log,clean_message,log_level,service,timestamp,structured_fields,label,confidence,classified_by


In [58]:
REGEX_RULES = [
    # ================= Security =================
    {
        "level": "ERROR",
        "pattern": r"authentication failure",
        "confidence": 0.95,
        "sources": ["linux"]
    },
    {
        "level": "WARNING",
        "pattern": r"check pass; user unknown",
        "confidence": 0.96,
        "sources": ["linux"]
    },

    # ================= Core Dumps =================
    {
        "level": "CRITICAL",
        "pattern": r"generating core\.\d+",
        "confidence": 0.98,
        "sources": ["bgl"]
    },

    # ================= HDFS =================
    {
        "level": "INFO",
        "pattern": r"packetresponder \d+ for block blk_.* terminating",
        "confidence": 0.90,
        "sources": ["hdfs"]
    },
    {
        "level": "INFO",
        "pattern": r"block\* namesystem\.addstoredblock",
        "confidence": 0.88,
        "sources": ["hdfs"]
    },
    {
        "level": "INFO",
        "pattern": r"received block blk_",
        "confidence": 0.87,
        "sources": ["hdfs"]
    },
    {
        "level": "INFO",
        "pattern": r"block\* namesystem\.allocateblock",
        "confidence": 0.87,
        "sources": ["hdfs"]
    },
]


In [59]:
import re


class RegexEngine:
    def __init__(self):
        # Compile regex patterns once (fast + production-safe)
        self.rules = [
            {
                **rule,
                "compiled": re.compile(rule["pattern"], re.IGNORECASE)
            }
            for rule in REGEX_RULES
        ]

    def classify(self, log: dict) -> dict:
        """
        Phase 1: Regex-based severity classification.

        Returns:
            {
                "level": <LEVEL or None>,
                "confidence": <float>,
                "classified_by": "regex" | "Unclassified"
            }
        """
        text = log.get("clean_message", "")
        source = log.get("source")

        for rule in self.rules:
            if source not in rule["sources"]:
                continue

            if rule["compiled"].search(text):
                return {
                    "level": rule["level"],
                    "confidence": rule["confidence"],
                    "classified_by": "regex"
                }

        # Explicitly unresolved — hand off to Phase 2
        return {
            "level": None,
            "confidence": 0.0,
            "classified_by": "Unclassified"
        }


In [61]:
import json

INPUT_PATH = "./unified_logs.ndjson"
OUTPUT_PATH = "./regex_output.ndjson"

regex_engine = RegexEngine()

total_logs = 0
regex_matched = 0

with open(INPUT_PATH, "r") as fin, open(OUTPUT_PATH, "w") as fout:
    for line in fin:
        log = json.loads(line)
        total_logs += 1

        # Phase 1: Regex severity classification
        result = regex_engine.classify(log)

        # Always write explicit fields (no ambiguity downstream)
        log["level"] = result.get("level")
        log["confidence"] = result.get("confidence", 0.0)
        log["classified_by"] = result.get("classified_by")

        if log["classified_by"] == "regex":
            regex_matched += 1

        fout.write(json.dumps(log) + "\n")

print(
    f"[PHASE 1 COMPLETE] "
    f"Regex classified {regex_matched}/{total_logs} logs "
    f"({regex_matched / max(total_logs, 1):.2%})"
)


[PHASE 1 COMPLETE] Regex classified 1330/10000 logs (13.30%)


In [62]:
df=pd.read_json('regex_output.ndjson',lines=True)

In [66]:
df.head()

,source,raw_log,clean_message,log_level,service,timestamp,structured_fields,label,level,confidence,classified_by
0,mac,Jul 1 09:00:55 calvisitor-10-105-160-95 kerne...,iothunderboltswitch<<NUM>>(0x0)::listenercallb...,INFO,kernel,2026-07-01 09:00:55,"{'host': 'calvisitor-10-105-160-95', 'pid': '0...",NaN,None,0.0,Unclassified
1,mac,Jul 1 09:01:05 calvisitor-10-105-160-95 com.a...,thermal pressure state: <NUM> memory pressure ...,WARN,com.apple.CDScheduler,2026-07-01 09:01:05,"{'host': 'calvisitor-10-105-160-95', 'pid': '43'}",NaN,None,0.0,Unclassified
2,mac,Jul 1 09:01:06 calvisitor-10-105-160-95 QQ[10...,fa||url||taskid[<NUM>] dealloc,INFO,QQ,2026-07-01 09:01:06,"{'host': 'calvisitor-10-105-160-95', 'pid': '1...",NaN,None,0.0,Unclassified
3,mac,Jul 1 09:02:26 calvisitor-10-105-160-95 kerne...,arpt: <NUM>.<NUM>: airport_brcm43xx::syncpower...,INFO,kernel,2026-07-01 09:02:26,"{'host': 'calvisitor-10-105-160-95', 'pid': '0...",NaN,None,0.0,Unclassified
4,mac,Jul 1 09:02:26 authorMacBook-Pro kernel[0]: A...,arpt: <NUM>.<NUM>: airport_brcm43xx::platformw...,INFO,kernel,2026-07-01 09:02:26,"{'host': 'authorMacBook-Pro', 'pid': '0', 'com...",NaN,None,0.0,Unclassified
